<a href="https://colab.research.google.com/github/CienciaDatosUdea/005_CCA_Estudiantes/blob/main/Laboratorios/03_Lab_naive_bayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Laboratorio: Naive Bayes Bernoulli para detectar spam

## Objetivo
Construir desde cero un clasificador **Naive Bayes Bernoulli** a partir de un corpus pequeño de correos.

Al terminar debes poder explicar y calcular:

$
P(Y),\qquad P(X_i\mid Y),\qquad P(X\mid Y),\qquad P(Y\mid X)
$

y comprender por qué la hipótesis

$
X_i \perp X_j\mid Y
$

reduce drásticamente la complejidad del modelo.

## Contexto

Queremos clasificar un correo como

$
Y=1:\ \text{spam}, \qquad Y=0:\ \text{normal}.
$

Usaremos cinco palabras del vocabulario:


$V=\{\text{dinero, gratis, premio, proyecto, reunion}\}.$

Cada correo se representa por

$X=(X_1,\ldots,X_5),$

donde

$
X_i=\begin{cases}
1 & \text{si aparece la palabra }i,\\
0 & \text{si no aparece.}
\end{cases}
$
Por ejemplo, "dinero gratis" se representa como $(1,1,0,0,0).$


In [4]:
corpus = [
    ("spam",   "gana dinero gratis"),
    ("spam",   "dinero gratis ahora"),
    ("spam",   "premio dinero gratis"),
    ("spam",   "gana premio ahora"),
    ("spam",   "en la reunion habra dinero gratis"),
    ("normal",  "daremos un premio despues de la reunion"),
    ("normal", "reunion de proyecto"),
    ("normal", "proyecto para mañana"),
    ("normal", "reunion mañana"),
    ("normal", "informe del proyecto"),
    ("normal", "informe del proyecto"),
]

vocabulario = ["dinero", "gratis", "premio", "proyecto", "reunion"]

## Parte 1 — Construir el vector \(X\)

1. Escribe una función `vectorizar(texto, vocabulario)` que transforme un correo en un vector binario.
2. Vectoriza todos los correos del corpus.
3. Verifica manualmente al menos dos ejemplos.

Ejemplo esperado:


$\text{"gana dinero gratis"}\longrightarrow(1,1,0,0,0)$


In [5]:
#Librerias

import numpy as np
import matplotlib.pyplot as plt


In [6]:
def vectorizar(texto, vocabulario):
    vector = np.zeros(len(vocabulario))
    for i, palabra in enumerate(vocabulario):
        if palabra in texto:
            vector[i] = 1
    return vector

In [7]:
vectorizar("proyecto para mañana", vocabulario)

array([0., 0., 0., 1., 0.])

In [8]:
vectorizar("en la reunion habra dinero gratis", vocabulario)

array([1., 1., 0., 0., 1.])

## Parte 2 — Calcular el prior \(P(Y)\)

Calcula:


$P(Y=\text{spam}),\qquad P(Y=\text{normal}).$


Recuerda:

$
P(Y=y)=\frac{\#\text{correos de clase }y}{\#\text{correos totales}}.
$

In [9]:
P_spam = len([i for i in corpus if i[0] == 'spam']) / len(corpus)
P_spam

0.45454545454545453

In [10]:
P_normal = len([i for i in corpus if i[0] == 'normal']) / len(corpus)
P_normal

0.5454545454545454

## Parte 3 — Calcular $P(X_i=1\mid Y)$

Para cada palabra calcula su frecuencia dentro de cada clase. Por ejemplo:

$
P(X_{dinero}=1\mid Y=spam)
=\frac{\#\text{spam que contienen dinero}}{\#\text{spam}}.
$

Construye una tabla con las cinco palabras y ambas clases.

**Pregunta:** ¿qué ocurre si una palabra nunca aparece en una clase?

In [11]:
# TODO: calcula las probabilidades condicionales sin suavizado
N_spam =  len([i for i in corpus if i[0] == 'spam']) 
N_spam_dinero = len([i for i in corpus if i[0] == 'spam' and 'dinero' in i[1]]) 
P_spam_dinero = N_spam_dinero / N_spam
print("P(X = dinero|Y =spam) = ", P_spam_dinero)


P(X = dinero|Y =spam) =  0.8


In [12]:
clases = ["spam", "normal"]
for clase in clases:
    print(f"Probabilidades condicionales para la clase {clase}:")
    N_clase = len([i for i in corpus if i[0] == clase])
    
    for palabra in vocabulario:
        N_clase_palabra = len([i for i in corpus if i[0] == clase and palabra in i[1]])
        P_clase_palabra = N_clase_palabra / N_clase
        print(f"P(X = {palabra}|Y ={clase}) = {P_clase_palabra}")
    print("------"*8)    
    print("\n")

Probabilidades condicionales para la clase spam:
P(X = dinero|Y =spam) = 0.8
P(X = gratis|Y =spam) = 0.8
P(X = premio|Y =spam) = 0.4
P(X = proyecto|Y =spam) = 0.0
P(X = reunion|Y =spam) = 0.2
------------------------------------------------


Probabilidades condicionales para la clase normal:
P(X = dinero|Y =normal) = 0.0
P(X = gratis|Y =normal) = 0.0
P(X = premio|Y =normal) = 0.16666666666666666
P(X = proyecto|Y =normal) = 0.6666666666666666
P(X = reunion|Y =normal) = 0.5
------------------------------------------------




**Respuesta:** si una palabra nunca aparece en los correos de una clase, su probabilidad estimada es exactamente

$P(X_i=1\mid Y=y)=0.$

Como la verosimilitud es un producto, $P(x\mid Y=y)=\prod_i P(X_i=x_i\mid Y=y)$, basta con que un correo contenga esa palabra para que todo el producto sea 0, sin importar lo que digan las demás palabras.

En nuestro caso se tiene que $P(X_{dinero}=1\mid normal)=P(X_{gratis}=1\mid normal)=0$ y $P(X_{proyecto}=1\mid spam)=0$. De manera que cualquier correo con "dinero" nunca podría clasificarse como normal, aunque contenga también "reunion" y "proyecto".



## Parte 4 — Suavizado de Laplace

Para evitar probabilidades exactamente iguales a cero, usa suavizado de Laplace para variables Bernoulli:


$\hat P(X_i=1\mid Y=y)=\frac{N_{iy}+1}{N_y+2},$


donde \(N_{iy}\) es el número de correos de clase \(y\) que contienen la palabra \(i\), y \(N_y\) es el número total de correos de esa clase.

Calcula de nuevo la tabla.

In [13]:
# TODO: calcula probabilidades condicionales con Laplace

clases = ["spam", "normal"]

def contar(corpus, clase, palabra):
    return sum(1 for c, texto in corpus if c == clase and palabra in texto.split())

# N_y: número de correos de cada clase
N_y = {clase: sum(1 for c, _ in corpus if c == clase) for clase in clases}

# Sin suavizado (Parte 3) y con Laplace (Parte 4), guardadas en diccionarios para reutilizarlas
P_sin = {clase: {p: contar(corpus, clase, p) / N_y[clase] for p in vocabulario} for clase in clases}
P_lap = {clase: {p: (contar(corpus, clase, p) + 1) / (N_y[clase] + 2) for p in vocabulario} for clase in clases}

print(f"{'palabra':<10} | {'spam':>7} {'spam(L)':>8} | {'normal':>7} {'normal(L)':>9}")
print("-" * 50)
for p in vocabulario:
    print(f"{p:<10} | {P_sin['spam'][p]:>7.3f} {P_lap['spam'][p]:>8.3f} | "
          f"{P_sin['normal'][p]:>7.3f} {P_lap['normal'][p]:>9.3f}")


palabra    |    spam  spam(L) |  normal normal(L)
--------------------------------------------------
dinero     |   0.800    0.714 |   0.000     0.125
gratis     |   0.800    0.714 |   0.000     0.125
premio     |   0.400    0.429 |   0.167     0.250
proyecto   |   0.000    0.143 |   0.667     0.625
reunion    |   0.200    0.286 |   0.500     0.500



Laplace equivale a sumar a cada clase dos correos "imaginarios": uno que contiene la palabra y otro que no. Así ninguna probabilidad es 0 ni 1, y las estimaciones se acercan un poco a 0.5 (el efecto es mayor cuanto más pequeño es el corpus).

## Parte 5 — Clasificar un correo nuevo

Clasifica:

> **"dinero gratis"**

Su vector es

$x=(1,1,0,0,0).$

Bajo Naive Bayes Bernoulli:

$P(x\mid Y=y)=\prod_iP(X_i=x_i\mid Y=y).$

Recuerda que para una palabra ausente:

$P(X_i=0\mid Y=y)=1-P(X_i=1\mid Y=y).$


Calcula los scores conjuntos:

$S_y=P(Y=y)P(x\mid Y=y),$

y finalmente:

$P(Y=y\mid x)=\frac{S_y}{S_{spam}+S_{normal}}.$

Decide la clase del correo.

In [15]:
# TODO: como computar likelihood, score conjunto y posterior para "dinero gratis"
x = vectorizar("dinero gratis", vocabulario)   # (1, 1, 0, 0, 0)
prior = {"spam": P_spam, "normal": P_normal}

def likelihood(x, P_cond):
    """P(x|Y=y) = prod_i P(X_i = x_i | Y=y), usando P(X_i=0|y) = 1 - P(X_i=1|y)."""
    L = 1.0
    for xi, palabra in zip(x, vocabulario):
        p1 = P_cond[palabra]
        L *= p1 if xi == 1 else (1 - p1)
    return L

def clasificar(x, P):
    L = {c: likelihood(x, P[c]) for c in clases}       # verosimilitudes P(x|y)
    S = {c: prior[c] * L[c] for c in clases}           # scores conjuntos S_y
    Z = sum(S.values())                                # evidencia P(x)
    post = {c: S[c] / Z for c in clases}               # posteriores P(y|x)
    return L, S, post

print("x =", x)
for nombre, P in [("Con Laplace", P_lap)]:
    L, S, post = clasificar(x, P)
    print(f"\n=== {nombre} ===")
    for c in clases:
        print(f"P(x|{c:<6}) = {L[c]:.6f}   S_{c:<6} = {S[c]:.6f}   P({c}|x) = {post[c]:.4f}")
    print("Clase predicha:", max(post, key=post.get))


x = [1. 1. 0. 0. 0.]

=== Con Laplace ===
P(x|spam  ) = 0.178497   S_spam   = 0.081135   P(spam|x) = 0.9854
P(x|normal) = 0.002197   S_normal = 0.001199   P(normal|x) = 0.0146
Clase predicha: spam


## Parte 6 — Interpretación

Responde brevemente:

1. ¿Dónde se usa la hipótesis de independencia condicional?
2. ¿Por qué no necesitamos almacenar una probabilidad para cada uno de los \(2^5\) vectores posibles?
3. ¿Por qué Naive Bayes se considera un modelo **generativo** aunque aquí lo usemos para clasificar?

**1. ¿Dónde se usa la hipótesis de independencia condicional?**

En la factorización de la verosimilitud de la Parte 5. Por la regla de la cadena, en general

$P(x\mid Y)=\prod_i P(X_i=x_i\mid X_1,\ldots,X_{i-1},Y).$

La hipótesis $X_i\perp X_j\mid Y$ permite reemplazar cada factor por $P(X_i=x_i\mid Y)$, de modo que $P(x\mid Y)=\prod_i P(X_i=x_i\mid Y)$. Gracias a eso, en las Partes 3 y 4 bastó con estimar cada palabra por separado contando frecuencias dentro de cada clase.

**2. ¿Por qué no necesitamos una probabilidad para cada uno de los $2^5$ vectores?**

Sin la hipótesis, $P(X\mid Y=y)$ sería una distribución general sobre $2^5=32$ vectores: hacen falta $2^5-1=31$ parámetros por clase, es decir $2(2^5-1)=62$ en total. Además, con solo 11 correos la mayoría de esos 32 vectores nunca se observa, así que no se podrían estimar. Con la hipótesis basta una Bernoulli por palabra y por clase: $2\cdot 5=10$ parámetros (más el prior). El número de parámetros crece de forma lineal ($2d$) en lugar de exponencial ($2(2^d-1)$). Con un vocabulario realista de $d=10\,000$ palabras, la versión completa sería imposible.

**3. ¿Por qué es un modelo generativo?**

Porque modela la distribución conjunta de los datos, $P(X,Y)=P(Y)\,P(X\mid Y)$, es decir, describe cómo se generan los correos: primero se elige la clase con probabilidad $P(Y)$ y luego cada palabra aparece con probabilidad $P(X_i=1\mid Y)$. Con el modelo entrenado se podrían incluso muestrear correos sintéticos de spam o normales. La clasificación se obtiene después, de forma indirecta, aplicando el teorema de Bayes, $P(Y\mid X)\propto P(Y)P(X\mid Y)$. Un modelo discriminativo modela directamente $P(Y\mid X)$ sin describir cómo se distribuye $X$.